# Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import os

# Data

In [2]:
#os.getcwd()
os.chdir(r'c:\\Users\\arvin\\Documents\\Coding Project\\V4\\Algotrading_RL\\src\\data')
#os.getcwd()
df = pd.read_csv('BTCUSD_1m_2024-09-23.csv', index_col='time', usecols=['time', 'open', 'high', 'low', 'close', 'tick_volume'])
df.index = pd.to_datetime(df.index)

os.chdir(r'c:\\Users\\arvin\\Documents\\Coding Project\\V4\\Algotrading_RL\\src\\brute_force')

price_column = 'close'
date_split = "2024-10-1"

df_train = df[:date_split]
df_test = df[date_split:]

tf = '1min'

# MACD Calculator class

In [3]:
class MACD_calculator:
    def __init__(self, df, price_column):
        self.df = df
        self.price_column = price_column
        self.macd = None
        self.signal_line = None
        self.position = None
        
    def traditional_MACD(self):
        """
        Calculate the traditional MACD and signal line without storing intermediate EMAs
        """
        # Use single-pass operations where possible
        price_df = self.df[self.price_column]
        
        # Calculate EMAs and MACD in a single step, optimizing memory usage
        ema_12 = price_df.ewm(span=12, adjust=False).mean()
        ema_26 = price_df.ewm(span=26, adjust=False).mean()
        self.macd = ema_12 - ema_26
        
        # Calculate the signal line directly from the MACD
        self.signal_line = self.macd.ewm(span=9, adjust=False).mean()
        
        return self  # Return self to allow method chaining

    def optimize_MACD(self, coff_ema_short, coff_ema_long, coff_signal_line):
        """
        Optimize MACD calculation with custom coefficients for short, long EMAs and the signal line.
        """
        price_df = self.df[self.price_column]
        
        # Calculate optimized EMAs with custom coefficients
        ema_short = price_df.ewm(span=round(coff_ema_short), adjust=False).mean()
        ema_long = price_df.ewm(span=round(coff_ema_long), adjust=False).mean()
        self.macd = ema_short - ema_long
        
        # Calculate the signal line with the custom coefficient
        self.signal_line = self.macd.ewm(span=round(coff_signal_line), adjust=False).mean()
        
        return self  # Return self for chaining
    
    def generate_signals(self):
        """
        Generate buy (1) and sell (0) signals based on the MACD crossover with the signal line.
        """
        if self.macd is None or self.signal_line is None:
            raise ValueError("MACD and Signal Line must be calculated before generating signals.")
        
        # Efficient vectorized signal generation using numpy
        signal = np.where(self.macd > self.signal_line, 1, 0)  # 1: Buy, 0: Sell
        self.position = np.diff(signal, prepend=0)  # Calculate positions (buy/sell transitions)
        
        return self  # Return self for chaining
    
    def number_of_trades(self):
        """
        Count the number of buy and sell trades based on the generated signals.
        """
        if self.position is None:
            raise ValueError("Signals must be generated before counting trades.")
        
        # Efficient count of buy and sell trades using numpy's boolean indexing
        buy_count = np.sum(self.position == 1)  # Buy trades (where position == 1)
        sell_count = np.sum(self.position == -1)  # Sell trades (where position == -1)
        
        print(f"The number of buy signals is: {buy_count}")
        print(f"The number of sell signals is: {sell_count}")
        
        return self  # Return self for chaining
    
# Outputs
# .macd and .signal_line for traditional macd and optimize macd
# .position for generate signals
# number of trade with produce automatically

# Trade Analyzer class

In [4]:
class TradeAnalyzer:
    def __init__(self, df, positions, price_column, sell_fee=0.115, buy_fee = 0.115):
        self.df = df.copy()
        self.df["Positions"] = positions  # Attach the positions to the df
        self.price_column = price_column  # Store the price column name
        self.sell_fee_percent = sell_fee / 100  # Convert fee to decimal
        self.buy_fee_percent = buy_fee / 100
        self.returns = []

    def calculate_return(self):
        # Create buy/sell columns using np.where based on positions
        buy = pd.Series(np.where(self.df["Positions"] == 1, self.df[self.price_column], np.nan))
        sell = pd.Series(np.where(self.df["Positions"] == -1, self.df[self.price_column], np.nan))
        
        # Forward-fill the buy price to match the next sell signal
        buy = buy.ffill()  # Ensure buys are forward-filled until the next sell
        
        # Create valid trades dfFrame with non-null buy-sell pairs
        trades = pd.DataFrame({'Buy': buy, 'Sell': sell}).dropna(subset=['Sell'])

        if trades.empty:
            print("No complete buy-sell pairs found.")
            return

        # Calculate raw trade returns
        trade_returns = trades['Sell'] - trades['Buy']

        # Apply fee on both buy and sell transactions
        net_returns = ((trade_returns - (trades['Buy'] * self.buy_fee_percent) - (trades['Sell'] * self.sell_fee_percent)) / trades['Buy']) * 100

        # Store the returns for each trade
        self.returns = net_returns.tolist()

    def print_returns(self):
        """
        Print each trade's return and the total return.
        """
        if not self.returns:
            print("No complete buy-sell pairs found.")
            return
        
        for i, trade_return in enumerate(self.returns, 1):
            print(f"Trade {i}: Return = {trade_return:.2f}%")
        
        total_return = np.sum(self.returns)  # Sum of all returns
        print(f"Total Return for all trades with fees using traditional MACD: {total_return:.2f}%")

    def get_total_return(self):
        """
        Return the sum of all returns for the trades.
        """
        if not self.returns:
            return 0
        return round(np.sum(self.returns),2)


# brute force

In [ ]:
price_type = 'close'
all_return1 = []
for i in range(2,5):
    for j in range(3,50):
        for k in range(2,40):
            if i >= j:
                break
            else:
                
                macd = MACD_calculator(df, price_column=price_type).optimize_MACD(i, j, k).generate_signals()
                positions = macd.position
                trade_analyzer = TradeAnalyzer(df, positions, price_column=price_type)
                trade_analyzer.calculate_return()
                trade_analyzer.print_returns()
                
                total_return = trade_analyzer.get_total_return()
                all_return1.append((i, j, k, total_return))

                #all_return[i,j,k] = 

all_return2 = []
for i in range(5,10):
    for j in range(3,50):
        for k in range(2,40):
            if i >= j:
                break
            else:
                
                macd = MACD_calculator(df, price_column=price_type).optimize_MACD(i, j, k).generate_signals()
                positions = macd.position
                trade_analyzer = TradeAnalyzer(df, positions, price_column=price_type)
                trade_analyzer.calculate_return()
                trade_analyzer.print_returns()
                
                total_return = trade_analyzer.get_total_return()
                all_return2.append((i, j, k, total_return))

all_return3 = []
for i in range(10,20):
    for j in range(3,50):
        for k in range(2,40):
            if i >= j:
                break
            else:
                
                macd = MACD_calculator(df, price_column=price_type).optimize_MACD(i, j, k).generate_signals()
                positions = macd.position
                trade_analyzer = TradeAnalyzer(df, positions, price_column=price_type)
                trade_analyzer.calculate_return()
                trade_analyzer.print_returns()
                
                total_return = trade_analyzer.get_total_return()
                all_return3.append((i, j, k, total_return))

all_return4 = []
for i in range(20,30):
    for j in range(3,50):
        for k in range(2,40):
            if i >= j:
                break
            else:
                
                macd = MACD_calculator(df, price_column=price_type).optimize_MACD(i, j, k).generate_signals()
                positions = macd.position
                trade_analyzer = TradeAnalyzer(df, positions, price_column=price_type)
                trade_analyzer.calculate_return()
                trade_analyzer.print_returns()
                
                total_return = trade_analyzer.get_total_return()
                all_return4.append((i, j, k, total_return))


all_return5 = []
for i in range(30,50):
    for j in range(3,50):
        for k in range(2,40):
            if i >= j:
                break
            else:
                
                macd = MACD_calculator(df, price_column=price_type).optimize_MACD(i, j, k).generate_signals()
                positions = macd.position
                trade_analyzer = TradeAnalyzer(df, positions, price_column=price_type)
                trade_analyzer.calculate_return()
                trade_analyzer.print_returns()
                
                total_return = trade_analyzer.get_total_return()
                all_return5.append((i, j, k, total_return))



all_returns = all_return1 + all_return2 + all_return3 + all_return4 + all_return5
sorted_all_returns = sorted(all_returns, key=lambda x: x[3], reverse=True)

# Extract the best parameters from the sorted returns
best_parameters = sorted_all_returns[0]  # This gives the tuple (i, j, k, total_return)
best_ema_short = best_parameters[0]
best_ema_long = best_parameters[1]
best_signal_line = best_parameters[2]

print(f"Best MACD Parameters -> EMA Short: {best_ema_short}, EMA Long: {best_ema_long}, Signal Line: {best_signal_line}")
